# Verify Class Mapping for `vit_ticks_fold1.pth`

目的: 推論側 (`django_prediction_API/prediction/views.py`) の `CLASS_NAMES` の順序が、実際の学習時のクラス index と合っているかを実測で確認する。

やり方: 種類が分かっている画像を流し、各 index の softmax 確率を出す。最も反応する index = モデルが内部で割り当てている真の index。

## 1. Drive マウント & 依存インストール

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q torch torchvision Pillow

## 2. モデルロード

`CLS_WEIGHTS` を Drive 上の `vit_ticks_fold1.pth` のパスに書き換える。

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from torchvision import models, transforms
from PIL import Image
from pathlib import Path

# --- TODO: 自分の環境に合わせて書き換える ---
CLS_WEIGHTS = "/content/drive/MyDrive/path/to/vit_ticks_fold1.pth"
# --------------------------------------------

NUM_CLASSES = 4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 現在の views.py のラベル (Unicode 順)。これが正しいかをこれから検証する。
CURRENT_LABELS = ["カクマダニ", "タカサゴキララマダニ", "チマダニ", "マダニ"]

def build_model():
    model = models.vit_b_16(weights=None)
    in_features = model.heads.head.in_features
    model.heads.head = nn.Linear(in_features, NUM_CLASSES)
    state = torch.load(CLS_WEIGHTS, map_location=DEVICE)
    model.load_state_dict(state, strict=True)
    return model.to(DEVICE).eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

model = build_model()
print("Model loaded on", DEVICE)

## 3. 検証用画像を指定

各クラスにつき、種が確実に分かっている画像を 3〜5 枚指定。

**注意**: 本番の `views.py` は YOLO で切り出してから分類器に渡しているため、ここでも YOLO 切り出し済み（マダニ単体が写っている）クロップ画像を使うこと。元画像をそのまま渡すと結果がぶれる。

In [ ]:
TEST_IMAGES = {
    "キララマダニ (Amblyomma)": [
        "/content/drive/MyDrive/tick_test/amblyomma_1.jpg",
        "/content/drive/MyDrive/tick_test/amblyomma_2.jpg",
    ],
    "チマダニ (Haemaphysalis)": [
        "/content/drive/MyDrive/tick_test/haemaphysalis_1.jpg",
    ],
    "カクマダニ": [
        "/content/drive/MyDrive/tick_test/kaku_1.jpg",
    ],
    "マダニ (Ixodes)": [
        "/content/drive/MyDrive/tick_test/ixodes_1.jpg",
    ],
}

## 4. 推論実行 & 結果表示

In [ ]:
def predict_probs(img_path):
    img = Image.open(img_path).convert("RGB")
    tensor = transform(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = model(tensor)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    return probs

print("=" * 90)
print("各画像の softmax 確率（クラス index 別）")
print("=" * 90)
print(f"{'GT':<28} {'idx0':>8} {'idx1':>8} {'idx2':>8} {'idx3':>8}  argmax  file")
print("-" * 90)

agg = {gt: np.zeros(NUM_CLASSES) for gt in TEST_IMAGES}
cnt = {gt: 0 for gt in TEST_IMAGES}

for gt, paths in TEST_IMAGES.items():
    for p in paths:
        probs = predict_probs(p)
        agg[gt] += probs
        cnt[gt] += 1
        argmax_idx = int(np.argmax(probs))
        print(f"{gt:<28} "
              f"{probs[0]:>8.3f} {probs[1]:>8.3f} {probs[2]:>8.3f} {probs[3]:>8.3f}  "
              f"idx{argmax_idx}   {Path(p).name}")

print()
print("=" * 90)
print("GT ラベル別 平均確率 (← これが結論)")
print("=" * 90)
print(f"{'GT':<28} {'idx0':>8} {'idx1':>8} {'idx2':>8} {'idx3':>8}  → argmax (現在の表示ラベル)")
print("-" * 90)
for gt in TEST_IMAGES:
    if cnt[gt] == 0:
        continue
    mean_probs = agg[gt] / cnt[gt]
    argmax_idx = int(np.argmax(mean_probs))
    print(f"{gt:<28} "
          f"{mean_probs[0]:>8.3f} {mean_probs[1]:>8.3f} {mean_probs[2]:>8.3f} {mean_probs[3]:>8.3f}  "
          f"→ idx{argmax_idx} ({CURRENT_LABELS[argmax_idx]})")

print()
print("読み方:")
print("  各 GT 行で最も高い確率を持つ index = モデルがその種に割り当てている真の index")
print("  例: GT=キララマダニ で idx2 が最大なら、真のマッピングは CLASS_NAMES[2] = 'タカサゴキララマダニ'")
print("  4 クラス分そろえば views.py を修正できる")